# Distill Anything — guided tour

**Distill any model into one you own**: a teacher generates your dataset, a student trains on its logits or its words, a judge scores the result blind, and a benchmark prices it.

This notebook walks the whole lifecycle:

1. [Install & verify](#1) — `pip install distill-anything`, then a <1 min offline self-test
2. [Generate a dataset](#2) — a teacher model writes your training data
3. [Distill a student](#3) — white-box logit KD in three lines
4. [The report card](#4) — blind LLM-as-judge quality + efficiency vs the teacher
5. [Benchmark & chat](#5) — latency, tokens/s, memory, $/1K tokens
6. [API teachers](#6) — swap one string to use Claude / GPT / Ollama

Sections 2–5 download ~1GB of SmolLM2 models on first run and take ~15 min total on an Apple-Silicon MacBook (MPS), less on CUDA. Section 1 is instant and offline.

Repo: https://github.com/AIAnytime/distillanything · PyPI: https://pypi.org/project/distill-anything/

In [ ]:
# Install from PyPI. Either package manager works:
%pip install -q distill-anything
# ...or with uv, from a terminal:  uv pip install distill-anything

# Optional extras when you want API teachers/judges (used in section 6):
#   %pip install -q "distill-anything[anthropic]"   # Claude
#   %pip install -q "distill-anything[openai]"      # OpenAI / vLLM / Ollama

In [ ]:
import distillanything
from distillanything.hardware import device_summary

# The framework picks the best device automatically everywhere:
# CUDA (bf16 autocast) -> Apple Silicon MPS -> CPU
print("version:", distillanything.__version__)
print("device: ", device_summary())

<a id="1"></a>
## 1. Verify your machine (offline, <1 min)

`run_smoke()` builds **tiny random models** (a 2-layer student, a 4-layer teacher, a character-level tokenizer — nothing downloaded) and pushes them through the *entire* pipeline: tokenization with prompt masking → logit KD training → evaluation → benchmark. If the loss drops and it prints `PASS`, everything works on your hardware.

In [ ]:
from distillanything.smoke import run_smoke

smoke_results = run_smoke()  # equivalent CLI: distill smoke

<a id="2"></a>
## 2. Generate a training dataset with a teacher

Every knowledge source in Distill Anything is selected by **one spec string**:

| Spec | Backend | Distillation mode |
|---|---|---|
| `hf:HuggingFaceTB/SmolLM2-360M-Instruct` | local Hugging Face model | white-box (logits) |
| `claude` | Anthropic API | black-box (seqKD) |
| `openai:gpt-4o-mini` | OpenAI API | black-box |
| `ollama:llama3.2` | local Ollama server | black-box |

We seed a few prompts (in real use: **your domain** — support tickets, legal Q&A, your API's docs), let the teacher invent more (`expand_per_seed`), then answer all of them. Output is a deduped JSONL of `{prompt, response, teacher}` records.

In [ ]:
from pathlib import Path

from distillanything.data.generate import generate_dataset
from distillanything.teachers.registry import resolve_teacher

# A small local teacher so this cell needs no API key (~700MB first-time download).
TEACHER = "hf:HuggingFaceTB/SmolLM2-360M-Instruct"

Path("seeds.txt").write_text(
    "Explain the difference between a list and a tuple in Python.\n"
    "What are three ways to reduce cloud compute costs?\n"
    "Explain gradient descent using a hiking analogy.\n"
    "Write a short email declining a meeting politely.\n"
    "What is the difference between concurrency and parallelism?\n"
)

teacher = resolve_teacher(TEACHER)
records = generate_dataset(
    teacher,
    seeds_path="seeds.txt",
    out_path="data/train.jsonl",
    max_tokens=256,        # length of each teacher answer
    expand_per_seed=3,     # self-instruct: 3 new prompts invented per seed
)

records[0]  # peek at one training record

<a id="3"></a>
## 3. Distill a student

`Student.learn()` wires the whole thing: it resolves the teacher, tokenizes with **prompt masking** (only response tokens are supervised), and trains.

Because the teacher spec is `hf:`, this runs **white-box logit KD**: at every response position the student's next-token distribution is pulled toward the teacher's full distribution —

$$\mathcal{L} = \alpha \cdot T^2 \, \mathrm{KL}\big(p_T \,\|\, p_S\big) + (1-\alpha) \cdot \mathrm{CE}$$

**One rule:** logit KD needs student and teacher to *share a tokenizer* (same model family — here both are SmolLM2). Across tokenizers, use an API teacher and `mode="seqkd"` (plain fine-tuning on the teacher's text — the knowledge is already in the dataset from step 2).

In [ ]:
from distillanything import Student

student = Student("HuggingFaceTB/SmolLM2-135M-Instruct")

results = student.learn(
    teacher=TEACHER,               # hf: teacher -> logit KD chosen automatically
    dataset="data/train.jsonl",
    epochs=2,
    batch_size=2, grad_accum=8,    # effective batch of 16 sequences, laptop-safe
    lr=1e-4,
    output_dir="runs/notebook-student",
    # kind="reverse_kl",           # MiniLLM-style mode-seeking KD — try it!
    # top_k=256,                   # truncated KD: big memory saver on 50k vocabs
)

# perplexity: how surprised the student is by held-out teacher data
# teacher_agreement: how often student & teacher pick the same next token
results.get("eval")

<a id="4"></a>
## 4. The report card — "was it worth it?"

`build_report` answers the only question that matters: *did the student keep the teacher's quality at a fraction of the cost?*

The **judge** compares student vs reference answers **blind**, and judges every pair **twice with positions swapped** — a judge that always prefers "Answer A" nets out to a tie, and unparseable verdicts count as ties, never wins. Headline metric:

> **quality retention** = (wins + ties) / n — how often the student is at least as good.

Here the 360M teacher judges its own student (free, but coarse — small judges tie a lot). For sharp discrimination pass `judge="claude"` or `judge="openai:gpt-4o"`.

In [ ]:
from IPython.display import Markdown

from distillanything.eval.report import build_report

report_path = build_report(
    "runs/notebook-student",
    "data/train.jsonl",
    teacher=TEACHER,               # provides reference answers + the rival benchmark
    judge=TEACHER,                 # swap for "claude" if you have a key
    n=12,                          # held-out prompts to judge
    max_new_tokens=192,
    hardware_cost_per_hour=1.20,   # your $/hr -> $ per 1K tokens in the report
)

Markdown(report_path.read_text())  # REPORT.md, rendered inline

<a id="5"></a>
## 5. Benchmark & chat with your model

The distilled student is a **standard Hugging Face checkpoint** — `runs/notebook-student` loads with `AutoModelForCausalLM.from_pretrained`, deploys with vLLM/llama.cpp/anything.

In [ ]:
# p50/p95 latency over repeated runs, throughput, memory, and serving cost
print(student.benchmark(n_runs=5, hardware_cost_per_hour=1.20))

# Quick vibe check (prompts are rendered through the model's chat template)
print(student.generate("Explain gradient descent using a hiking analogy."))

<a id="6"></a>
## 6. Frontier teachers: Claude / GPT / Ollama

The whole notebook re-runs with a frontier teacher by changing **one string**. API teachers are black-box (no logits), so `Student.learn` automatically switches to **seqKD** — and if your dataset has prompts without responses, the teacher fills them in before training.

In [ ]:
import os

# Guarded so the notebook runs top-to-bottom without secrets.
if os.environ.get("ANTHROPIC_API_KEY"):
    # %pip install -q "distill-anything[anthropic]"
    claude_student = Student("HuggingFaceTB/SmolLM2-135M-Instruct")
    claude_student.learn(
        teacher="claude",                  # default model: claude-opus-4-8
        dataset="seeds.txt",               # prompts only -> Claude writes the answers
        output_dir="runs/claude-student",
    )
else:
    print("Set ANTHROPIC_API_KEY to distill from Claude — everything else is identical.")

## Where to go next

- **Scripts** — the same flow as standalone files: [`01_generate_dataset.py`](01_generate_dataset.py) → [`02_train_student.py`](02_train_student.py) → [`03_evaluate_and_report.py`](03_evaluate_and_report.py)
- **Recipes** — reproducible YAML runs for the CLI: [`recipes/`](../recipes) (`distill train recipes/mac-small.yaml`)
- **Tests as living docs** — the suite runs offline in seconds and doubles as usage examples: [`tests/test_losses.py`](../tests/test_losses.py) (KD math contracts), [`tests/test_judge.py`](../tests/test_judge.py) (how to fake a Teacher), [`tests/test_trainer.py`](../tests/test_trainer.py) (training with tiny models)
- **Contributing** — [`CONTRIBUTING.md`](../CONTRIBUTING.md); the test suite needs no GPU, no keys, no network.